# Data Preprocessing for FFNN

### Import Library

In [1]:
# import Module

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# import preprocessing tools
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

### import dataset

In [ ]:
# import csv dataset 

df = pd.read_csv('../dataset/datasetml_2026.csv')

df.head()

,cgpa,backlogs,college_tier,country,university_ranking_band,internship_count,aptitude_score,communication_score,specialization,industry,internship_quality_score,placement_status
0,7.397371,1,Tier 2,Canada,100-300,2,53.574150,64.177062,Data Science,Consulting,5.481450,Placed
1,6.889389,0,Tier 3,UK,300+,1,60.687750,88.346052,Data Science,Consulting,4.625099,Placed
2,7.518151,0,Tier 1,UK,100-300,2,64.568750,69.493171,Cybersecurity,Healthcare,5.227939,Placed
3,8.218424,0,Tier 2,UK,100-300,3,73.461500,78.204854,AI/ML,Tech,5.150674,Placed
4,6.812677,1,Tier 2,USA,100-300,4,86.518121,44.680881,Data Science,Consulting,3.888824,Placed


### Simple EDA

In [5]:
# Simple EDA
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   cgpa                      10000 non-null  float64
 1   backlogs                  10000 non-null  int64  
 2   college_tier              10000 non-null  str    
 3   country                   10000 non-null  str    
 4   university_ranking_band   10000 non-null  str    
 5   internship_count          10000 non-null  int64  
 6   aptitude_score            10000 non-null  float64
 7   communication_score       10000 non-null  float64
 8   specialization            10000 non-null  str    
 9   industry                  10000 non-null  str    
 10  internship_quality_score  10000 non-null  float64
 11  placement_status          10000 non-null  str    
dtypes: float64(4), int64(2), str(6)
memory usage: 937.6 KB


In [6]:
df.describe()

,cgpa,backlogs,internship_count,aptitude_score,communication_score,internship_quality_score
count,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000
mean,6.998290,1.248100,1.49930,69.877531,65.158600,5.021436
std,0.802606,1.149904,1.20289,14.700532,14.740446,1.505975
min,4.000000,0.000000,0.00000,30.000000,30.000000,1.000000
25%,6.461928,0.000000,1.00000,59.880399,55.112244,4.012656
50%,6.997924,1.000000,1.00000,70.097368,65.006484,5.017335
75%,7.536865,2.000000,2.00000,80.213934,75.277248,6.031400
max,10.000000,6.000000,5.00000,100.000000,100.000000,10.000000


In [7]:
# null check
df.isnull().sum()

cgpa                        0
backlogs                    0
college_tier                0
country                     0
university_ranking_band     0
internship_count            0
aptitude_score              0
communication_score         0
specialization              0
industry                    0
internship_quality_score    0
placement_status            0
dtype: int64

In [8]:
# cek balance class
counts = df['placement_status'].value_counts()
print(counts)
if counts.min() / counts.max() < 0.9:
    print("class imbalance")

placement_status
Placed        6153
Not Placed    3847
Name: count, dtype: int64
class imbalance


### Features target separation

In [9]:
X = df.drop('placement_status', axis=1)
y = df['placement_status'].map({'Placed': 1, 'Not Placed': 0})

In [10]:
X.shape, y.shape

((10000, 11), (10000,))

### Data Encoding

dataset clean dan dari data terlihat ada numeric dan kategorikal
langkah selanjutnya melakukan encoding untuk data nominal dan ordinal

a. Nominal (one hot encoding)
- country
- specialization
- industry

b. ordinal (Ordinal Encoding)
- college_tier
- university_ranking_band


In [11]:
# One-hot encoding untuk nominal
nominal_cols = ["country", "specialization", "industry"]
X_encoded = pd.get_dummies(X, columns=nominal_cols, drop_first=True)
X_encoded.head()

,cgpa,backlogs,college_tier,university_ranking_band,internship_count,aptitude_score,communication_score,internship_quality_score,country_Germany,country_India,...,country_USA,specialization_Cloud,specialization_Core CS,specialization_Cybersecurity,specialization_Data Science,industry_Finance,industry_Healthcare,industry_Manufacturing,industry_Other,industry_Tech
0,7.397371,1,Tier 2,100-300,2,53.574150,64.177062,5.481450,False,False,...,False,False,False,False,True,False,False,False,False,False
1,6.889389,0,Tier 3,300+,1,60.687750,88.346052,4.625099,False,False,...,False,False,False,False,True,False,False,False,False,False
2,7.518151,0,Tier 1,100-300,2,64.568750,69.493171,5.227939,False,False,...,False,False,False,True,False,False,True,False,False,False
3,8.218424,0,Tier 2,100-300,3,73.461500,78.204854,5.150674,False,False,...,False,False,False,False,False,False,False,False,False,True
4,6.812677,1,Tier 2,100-300,4,86.518121,44.680881,3.888824,False,False,...,True,False,False,False,True,False,False,False,False,False


In [12]:
# cek nilai unik pada ordinal
print("College Tier:", X['college_tier'].unique())
print("University Ranking Band:", X['university_ranking_band'].unique())


College Tier: <StringArray>
['Tier 2', 'Tier 3', 'Tier 1']
Length: 3, dtype: str
University Ranking Band: <StringArray>
['100-300', '300+', 'Top 100']
Length: 3, dtype: str


In [13]:
# Ordinal encoding untuk ordinal
ordinal_cols = ["college_tier", "university_ranking_band"]
ordinal_encoder = OrdinalEncoder(
    categories=[['Tier 1', 'Tier 2', 'Tier 3'], ['Top 100','100-300', '300+']],
    handle_unknown='use_encoded_value',
    unknown_value=-1
)
X_encoded[ordinal_cols] = ordinal_encoder.fit_transform(X_encoded[ordinal_cols])
X_encoded.head()


,cgpa,backlogs,college_tier,university_ranking_band,internship_count,aptitude_score,communication_score,internship_quality_score,country_Germany,country_India,...,country_USA,specialization_Cloud,specialization_Core CS,specialization_Cybersecurity,specialization_Data Science,industry_Finance,industry_Healthcare,industry_Manufacturing,industry_Other,industry_Tech
0,7.397371,1,1.0,1.0,2,53.574150,64.177062,5.481450,False,False,...,False,False,False,False,True,False,False,False,False,False
1,6.889389,0,2.0,2.0,1,60.687750,88.346052,4.625099,False,False,...,False,False,False,False,True,False,False,False,False,False
2,7.518151,0,0.0,1.0,2,64.568750,69.493171,5.227939,False,False,...,False,False,False,True,False,False,True,False,False,False
3,8.218424,0,1.0,1.0,3,73.461500,78.204854,5.150674,False,False,...,False,False,False,False,False,False,False,False,False,True
4,6.812677,1,1.0,1.0,4,86.518121,44.680881,3.888824,False,False,...,True,False,False,False,True,False,False,False,False,False


### Split Dataset

In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

print(f"X_train shape   : {X_train.shape}")
print(f"X_test shape    : {X_test.shape}")
print(f"y_train shape   : {y_train.shape}")
print(f"y_test shape    : {y_test.shape}")

# Cek class distribution pada train dan test
print("\nClass Dist Train Eval")
print("Train set:")
print(y_train.value_counts())
print("\nTest set:")
print(y_test.value_counts())

X_train shape   : (8000, 21)
X_test shape    : (2000, 21)
y_train shape   : (8000,)
y_test shape    : (2000,)

Class Dist Train Eval
Train set:
placement_status
1    4922
0    3078
Name: count, dtype: int64

Test set:
placement_status
1    1231
0     769
Name: count, dtype: int64


### Handle Imbalance (SMOTE)

In [15]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nClass Dist Train Eval After SMOTE")
print("Train set:")
print(y_train_smote.value_counts())


Class Dist Train Eval After SMOTE
Train set:
placement_status
1    4922
0    4922
Name: count, dtype: int64


### Scaller

In [16]:
scaler = StandardScaler()

# fit hanya buat train
X_train_scaled = scaler.fit_transform(X_train_smote)

# transform test
X_test_scaled = scaler.transform(X_test)

### NumPy Array Transform and Final Packaging

In [17]:
X_train_final = X_train_scaled
y_train_final = np.array(y_train_smote)
X_test_final = X_test_scaled
y_test_final = np.array(y_test)

In [18]:
X_train_final

array([[ 0.92436541, -0.24619835, -1.30896522, ..., -0.49892043,
        -0.48122513, -0.4682117 ],
       [-0.91856064, -0.24619835,  1.33598463, ..., -0.49892043,
        -0.48122513,  2.13578602],
       [-0.45925646, -1.13200848,  0.01350971, ..., -0.49892043,
        -0.48122513, -0.4682117 ],
       ...,
       [-0.8843968 , -1.13200848, -1.30896522, ..., -0.49892043,
        -0.48122513,  2.13578602],
       [-1.27338126,  0.63961179,  0.01350971, ..., -0.49892043,
        -0.48122513, -0.4682117 ],
       [-1.95354808,  1.52542192,  0.01350971, ...,  2.00432763,
        -0.48122513, -0.4682117 ]], shape=(9844, 21))

In [19]:
y_train_final

array([1, 1, 1, ..., 0, 0, 0], shape=(9844,))

In [20]:
X_test_final

array([[-0.07459367, -0.24619835,  0.01350971, ...,  2.00432763,
        -0.48122513, -0.4682117 ],
       [-1.74270496,  0.63961179, -1.30896522, ..., -0.49892043,
         2.07802945, -0.4682117 ],
       [-1.47310744, -0.24619835,  0.01350971, ...,  2.00432763,
        -0.48122513, -0.4682117 ],
       ...,
       [ 0.56014469, -0.24619835,  0.01350971, ..., -0.49892043,
        -0.48122513,  2.13578602],
       [ 0.76873032,  1.52542192, -1.30896522, ..., -0.49892043,
        -0.48122513, -0.4682117 ],
       [-0.18543814,  1.52542192, -1.30896522, ..., -0.49892043,
        -0.48122513, -0.4682117 ]], shape=(2000, 21))

In [21]:
y_test_final

array([0, 0, 0, ..., 0, 0, 0], shape=(2000,))

### Export Dataset

In [ ]:
# Buat direktri kalo blom ada

os.makedirs('../dataset/classification', exist_ok=True)

np.save('../dataset/classification/X_train_final.npy', X_train_final)
np.save('../dataset/classification/X_test_final.npy', X_test_final)
np.save('../dataset/classification/y_train_final.npy', y_train_final)
np.save('../dataset/classification/y_test_final.npy', y_test_final)